In [ ]:
from scipy.optimize import root

p_reaction_extent = np.array([1.0, 5.0, 10.0, 20.0, 150.0])
pressure_extent_cases = p_reaction_extent

p_standard = 1.0  # bar

component_index = {
    component: i
    for i, component in enumerate(component_order)
}

N_key = N[:, :R_nu]
key_reaction_names = reaction_order[:R_nu]

print("Selected independent reactions:")
for reaction_name in key_reaction_names:
    print(reaction_name)

print("\nRank of selected reaction matrix:")
print(np.linalg.matrix_rank(N_key))

T_extent_values_all = reaction_results[key_reaction_names[0]]["T"]

T_extent_mask = (
    (T_extent_values_all >= T_plot_min_K)
    & (T_extent_values_all <= T_plot_max_K)
)

T_extent_values = T_extent_values_all[T_extent_mask]


def calculate_Kx_for_extent(reaction_name, pressure):
    """
    Calculate Kx(T,p) newly for the reaction extent calculation.

    K° is calculated from Delta_R G°:
    K° = exp(-Delta_R G° / RT)

    Kx is calculated from:
    Kx = K° * (p° / p)^Delta_nu
    """

    T_values = reaction_results[reaction_name]["T"][T_extent_mask]
    delta_G_standard = reaction_results[reaction_name]["delta_G"][T_extent_mask]

    K_standard = np.exp(
        -delta_G_standard / (R * T_values)
    )

    delta_nu = sum(reactions[reaction_name].values())

    Kx = K_standard * (p_standard / pressure)**delta_nu

    return Kx


Kx_extent_results = {}

for pressure in pressure_extent_cases:

    Kx_extent_results[pressure] = {}

    for reaction_name in key_reaction_names:

        Kx_extent_results[pressure][reaction_name] = calculate_Kx_for_extent(
            reaction_name,
            pressure
        )


def get_Kx_value(reaction_name, pressure, TT):
    """
    Get newly calculated Kx value for one reaction, pressure and temperature index.
    """

    return Kx_extent_results[pressure][reaction_name][TT]


def composition(xi, n_in):
    """
    Calculate outlet molar flows and outlet mole fractions.

    xi contains the reaction extents of the selected independent reactions.
    The component order is given by component_order.
    """

    n_out = n_in + N_key.dot(xi)
    n_total = np.sum(n_out)

    if n_total <= 0 or np.any(n_out <= 0):
        x_out = np.full(len(n_in), np.nan)
    else:
        x_out = n_out / n_total

    return n_out, x_out


def rxn_ext(xi, n_in, TT, pp):
    """
    Nonlinear equilibrium equations for the independent reactions.

    Residual form:
    ln(Q_x) - ln(K_x) = 0
    """

    n_out, x = composition(xi, n_in)

    if np.any(np.isnan(x)) or np.any(x <= 0):
        return np.ones(R_nu) * 1e6

    pressure = pressure_extent_cases[pp]

    x_CO2 = x[component_index["CO2"]]
    x_CH3OH = x[component_index["CH3OH"]]
    x_CH3OCH3 = x[component_index["CH3OCH3"]]
    x_H2 = x[component_index["H2"]]
    x_H2O = x[component_index["H2O"]]
    x_CO = x[component_index["CO"]]

    res = np.empty(R_nu)

    for j, reaction_name in enumerate(key_reaction_names):

        Kx = get_Kx_value(reaction_name, pressure, TT)

        if Kx <= 0:
            return np.ones(R_nu) * 1e6

        if reaction_name == "Reaction 1":
            res[j] = (
                np.log(x_CH3OH)
                + np.log(x_H2O)
                - np.log(x_CO2)
                - 3.0 * np.log(x_H2)
                - np.log(Kx)
            )

        elif reaction_name == "Reaction 2":
            res[j] = (
                np.log(x_CH3OH)
                - np.log(x_CO)
                - 2.0 * np.log(x_H2)
                - np.log(Kx)
            )

        elif reaction_name == "Reaction 3":
            res[j] = (
                np.log(x_CO)
                + np.log(x_H2O)
                - np.log(x_CO2)
                - np.log(x_H2)
                - np.log(Kx)
            )

        elif reaction_name == "Reaction 4":
            res[j] = (
                np.log(x_CH3OCH3)
                + np.log(x_H2O)
                - 2.0 * np.log(x_CH3OH)
                - np.log(Kx)
            )

    return res


xi = np.empty(
    (pressure_extent_cases.shape[0], T_extent_values.shape[0], R_nu)
)

n_out_results = np.empty(
    (pressure_extent_cases.shape[0], T_extent_values.shape[0], len(component_order))
)

x_out_results = np.empty(
    (pressure_extent_cases.shape[0], T_extent_values.shape[0], len(component_order))
)

solver_success = np.empty(
    (pressure_extent_cases.shape[0], T_extent_values.shape[0]),
    dtype=bool
)

solver_message = {}

for pp, pressure in enumerate(pressure_extent_cases):

    xi_guess = np.full(R_nu, 1e-4)
    solver_message[pressure] = []

    for TT in range(T_extent_values.shape[0]):

        solution = root(
            rxn_ext,
            xi_guess,
            args=(n_in, TT, pp),
            method="lm"
        )

        xi[pp, TT, :] = solution.x
        solver_success[pp, TT] = solution.success
        solver_message[pressure].append(solution.message)

        n_out, x_out = composition(solution.x, n_in)

        n_out_results[pp, TT, :] = n_out
        x_out_results[pp, TT, :] = x_out

        if solution.success and np.all(n_out > 0) and not np.any(np.isnan(x_out)):
            xi_guess = solution.x

print("Reaction extents calculated.")

for pp, pressure in enumerate(pressure_extent_cases):
    successful = np.sum(solver_success[pp, :])
    total = solver_success.shape[1]
    print(f"{pressure:.0f} bar: {successful}/{total} solver runs successful")


extent_results = {
    "T": T_extent_values,
    "pressures": pressure_extent_cases,
    "reaction_names": key_reaction_names,
    "xi": xi,
    "n_out": n_out_results,
    "x_out": x_out_results,
    "success": solver_success,
    "message": solver_message,
    "Kx": Kx_extent_results
}


pressure_colors = {
    1.0: "#d62728",
    5.0: "#1f77b4",
    10.0: "#ff7f0e",
    20.0: "#2ca02c",
    150.0: "#9467bd"
}

pressure_linestyles = {
    1.0: "-",
    5.0: "--",
    10.0: ":",
    20.0: "-.",
    150.0: (0, (3, 1, 1, 1))
}

T_plot_C = T_extent_values - 273.15

fig, axes = plt.subplots(
    1,
    R_nu,
    figsize=(5 * R_nu, 5),
    sharex=True
)

if R_nu == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for j, ax in enumerate(axes):

    reaction_name = key_reaction_names[j]

    for pp, pressure in enumerate(pressure_extent_cases):

        ax.plot(
            T_plot_C,
            xi[pp, :, j],
            label=f"{pressure:.0f} bar",
            color=pressure_colors.get(float(pressure), None),
            linestyle=pressure_linestyles.get(float(pressure), "-"),
            linewidth=2.2
        )

    ax.axhline(
        0,
        color="black",
        linewidth=1,
        linestyle="--"
    )

    ax.set_title(rf"{reaction_name}")
    ax.set_xlabel("Temperature / °C")
    ax.set_ylabel(r"Reaction extent $\xi_j$ / mol s$^{-1}$")
    ax.grid(True)
    ax.legend(title="Pressure")

fig.suptitle(
    r"Reaction extents over temperature",
    fontsize=16,
    y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
# Parameter study: optimal H2/CO2 ratio for DME yield based on CO2
# This uses the already implemented equilibrium solver functions:
# reaction_extent_residual()
# composition_from_extent()

# Smaller evaluation grid for faster runtime
H2_CO2_ratios = np.linspace(1.0, 10.0, 15)

T_ratio = np.linspace(
    273.15 + 200,
    273.15 + 350,
    25
)

p_ratio = np.array([
    20.0,
    50.0,
    100.0
])

# Initial guess for reaction extents
xi_guess_ratio = xi_guess_manual.copy()

# Storage arrays
# Shape: ratio, pressure, temperature
Y_DME_CO2_ratio = np.full(
    (len(H2_CO2_ratios), len(p_ratio), len(T_ratio)),
    np.nan
)

xi_ratio = np.full(
    (len(H2_CO2_ratios), R_nu, len(p_ratio), len(T_ratio)),
    np.nan
)

valid_ratio = np.zeros(
    (len(H2_CO2_ratios), len(p_ratio), len(T_ratio)),
    dtype=bool
)

residual_ratio = np.full(
    (len(H2_CO2_ratios), len(p_ratio), len(T_ratio)),
    np.nan
)

idx_CO2 = component_order.index("CO2")
idx_H2 = component_order.index("H2")
idx_DME = component_order.index("CH3OCH3")

eps = 1e-12

for ratio_idx, H2_CO2_ratio in enumerate(H2_CO2_ratios):

    # order: CO2, CH3OH, CH3OCH3, H2, H2O, CO
    n_in_ratio = np.zeros(len(component_order))
    n_in_ratio[idx_CO2] = 1.0
    n_in_ratio[idx_H2] = H2_CO2_ratio

    n_CO2_in = n_in_ratio[idx_CO2]
    n_DME_in = n_in_ratio[idx_DME]

    print(
        f"Calculating ratio {ratio_idx + 1}/{len(H2_CO2_ratios)}: "
        f"H2/CO2 = {H2_CO2_ratio:.2f}"
    )

    for p_idx, pressure in enumerate(p_ratio):

        xi_guess = xi_guess_ratio.copy()

        for T_idx, T_val in enumerate(T_ratio):

            trial_guesses = [
                xi_guess,
                xi_guess_ratio,
                np.zeros(R_nu),
                0.1 * xi_guess_ratio
            ]

            best_solution = None
            best_norm = np.inf
            best_physical = False

            for guess in trial_guesses:

                solution = root(
                    reaction_extent_residual,
                    guess,
                    args=(n_in_ratio, T_val, pressure),
                    method="lm"
                )

                xi_trial = solution.x

                n_out_trial, _ = composition_from_extent(
                    xi_trial,
                    n_in_ratio
                )

                res_trial = reaction_extent_residual(
                    xi_trial,
                    n_in_ratio,
                    T_val,
                    pressure
                )

                norm_trial = np.linalg.norm(res_trial)
                physical_trial = np.all(n_out_trial >= -eps)

                if solution.success and physical_trial and norm_trial < best_norm:
                    best_solution = xi_trial
                    best_norm = norm_trial
                    best_physical = True

            if best_solution is not None and best_norm < 1e-6 and best_physical:

                xi_ratio[ratio_idx, :, p_idx, T_idx] = best_solution
                valid_ratio[ratio_idx, p_idx, T_idx] = True
                residual_ratio[ratio_idx, p_idx, T_idx] = best_norm

                n_out_current, _ = composition_from_extent(
                    best_solution,
                    n_in_ratio
                )

                n_DME_out = n_out_current[idx_DME]
                n_DME_formed = n_DME_out - n_DME_in

                Y_DME_CO2_ratio[ratio_idx, p_idx, T_idx] = (
                    2.0 * n_DME_formed / n_CO2_in
                )

                # Continuation: use current solution as next guess
                xi_guess = best_solution


# Determine maximum DME yield over temperature for each ratio and pressure

Y_DME_CO2_max = np.full(
    (len(H2_CO2_ratios), len(p_ratio)),
    np.nan
)

T_at_Y_DME_CO2_max = np.full_like(
    Y_DME_CO2_max,
    np.nan
)

for ratio_idx in range(len(H2_CO2_ratios)):

    for p_idx in range(len(p_ratio)):

        Y_values = Y_DME_CO2_ratio[ratio_idx, p_idx, :]

        if np.any(~np.isnan(Y_values)):

            max_idx = np.nanargmax(Y_values)

            Y_DME_CO2_max[ratio_idx, p_idx] = Y_values[max_idx]
            T_at_Y_DME_CO2_max[ratio_idx, p_idx] = (
                T_ratio[max_idx] - 273.15
            )


DME_ratio_study_results = {
    "H2_CO2_ratios": H2_CO2_ratios,
    "T": T_ratio,
    "p": p_ratio,
    "Y_DME_CO2": Y_DME_CO2_ratio,
    "Y_DME_CO2_max": Y_DME_CO2_max,
    "T_at_Y_DME_CO2_max": T_at_Y_DME_CO2_max,
    "xi": xi_ratio,
    "valid": valid_ratio,
    "residual": residual_ratio
}


# Plot maximum DME yield over H2/CO2 ratio

pressure_colors = {
    20.0: "#2ca02c",
    50.0: "#9467bd",
    100.0: "#8c564b"
}

pressure_linestyles = {
    20.0: "-.",
    50.0: (0, (3, 1, 1, 1)),
    100.0: (0, (5, 2))
}

fig, ax = plt.subplots(figsize=(9, 6))

for p_idx, pressure in enumerate(p_ratio):

    pressure_float = float(pressure)

    ax.plot(
        H2_CO2_ratios,
        Y_DME_CO2_max[:, p_idx],
        label=f"{pressure_float:.0f} bar",
        color=pressure_colors.get(pressure_float, None),
        linestyle=pressure_linestyles.get(pressure_float, "-"),
        linewidth=2.2
    )

    if np.any(~np.isnan(Y_DME_CO2_max[:, p_idx])):

        optimum_idx = np.nanargmax(Y_DME_CO2_max[:, p_idx])

        ax.scatter(
            H2_CO2_ratios[optimum_idx],
            Y_DME_CO2_max[optimum_idx, p_idx],
            color=pressure_colors.get(pressure_float, None),
            s=55,
            zorder=3
        )

        print(
            f"{pressure_float:.0f} bar: "
            f"optimal H2/CO2 = {H2_CO2_ratios[optimum_idx]:.2f}, "
            f"max Y_DME_CO2 = {Y_DME_CO2_max[optimum_idx, p_idx]:.4f}, "
            f"T = {T_at_Y_DME_CO2_max[optimum_idx, p_idx]:.1f} °C"
        )

ax.set_title(
    r"Influence of $H_2/CO_2$ feed ratio on equilibrium DME yield"
)

ax.set_xlabel(r"$H_2/CO_2$ feed ratio / 1")
ax.set_ylabel(r"Maximum $Y_{DME,CO_2}$ / 1")

ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(title="Pressure")

plt.tight_layout()
plt.show()

In [ ]:
# Reaction extent calculation for optimal CO2/H2 inlet composition

# Choose pressure case used to determine the optimal ratio
# Example: choose 50 bar if available
target_pressure_for_optimum = 50.0

if "DME_ratio_study_results" in globals():

    H2_CO2_ratios_opt = DME_ratio_study_results["H2_CO2_ratios"]
    p_ratio_opt = DME_ratio_study_results["p"]
    Y_DME_CO2_max_opt = DME_ratio_study_results["Y_DME_CO2_max"]

    p_opt_idx = np.argmin(np.abs(p_ratio_opt - target_pressure_for_optimum))

    ratio_opt_idx = np.nanargmax(Y_DME_CO2_max_opt[:, p_opt_idx])

    optimal_H2_CO2_ratio = H2_CO2_ratios_opt[ratio_opt_idx]

else:
    # Manual fallback if ratio study was not run before
    optimal_H2_CO2_ratio = 3.0


print(f"Optimal H2/CO2 ratio used: {optimal_H2_CO2_ratio:.3f}")


# Define optimal inlet composition
# order: CO2, CH3OH, CH3OCH3, H2, H2O, CO
n_in_optimal = np.zeros(len(component_order))

idx_CO2 = component_order.index("CO2")
idx_H2 = component_order.index("H2")

n_in_optimal[idx_CO2] = 1.0
n_in_optimal[idx_H2] = optimal_H2_CO2_ratio


# Store as inlet stream dictionary for compatibility
optimal_inlet_streams = {
    "optimal inlet": n_in_optimal
}

optimal_stream_names = list(optimal_inlet_streams.keys())
n_optimal_streams = len(optimal_stream_names)


# Independent reactions
independent_reactions = reaction_order[:R_nu]

# Storage arrays
xi_optimal = np.full(
    (R_nu, len(p_equilibrium), len(T_equilibrium)),
    np.nan
)

xi_optimal_plot = np.full_like(
    xi_optimal,
    np.nan
)

residual_norm_optimal = np.full(
    (len(p_equilibrium), len(T_equilibrium)),
    np.nan
)

solver_success_optimal = np.zeros(
    (len(p_equilibrium), len(T_equilibrium)),
    dtype=bool
)

physical_solution_optimal = np.zeros(
    (len(p_equilibrium), len(T_equilibrium)),
    dtype=bool
)

valid_solution_optimal = np.zeros(
    (len(p_equilibrium), len(T_equilibrium)),
    dtype=bool
)

n_out_optimal = np.full(
    (len(component_order), len(p_equilibrium), len(T_equilibrium)),
    np.nan
)

x_out_optimal = np.full_like(
    n_out_optimal,
    np.nan
)


# Initial guesses
xi_guess_base = xi_guess_manual.copy()

fallback_guesses = [
    xi_guess_base,
    np.zeros(R_nu),
    reaction_extent,
    0.1 * reaction_extent
]


# Run solver for optimal inlet composition
n_in_equilibrium_optimal = n_in_optimal.copy()

for p_idx, pressure in enumerate(p_equilibrium):

    xi_guess = xi_guess_base.copy()

    for T_idx, T_val in enumerate(T_equilibrium):

        trial_guesses = [xi_guess] + fallback_guesses

        best_solution = None
        best_norm = np.inf
        best_success = False
        best_physical = False

        for guess in trial_guesses:

            solution = root(
                reaction_extent_residual,
                guess,
                args=(n_in_equilibrium_optimal, T_val, pressure),
                method="lm"
            )

            xi_trial = solution.x

            n_out_trial, _ = composition_from_extent(
                xi_trial,
                n_in_equilibrium_optimal
            )

            res_trial = reaction_extent_residual(
                xi_trial,
                n_in_equilibrium_optimal,
                T_val,
                pressure
            )

            norm_trial = np.linalg.norm(res_trial)
            physical_trial = np.all(n_out_trial >= -1e-12)

            if physical_trial and norm_trial < best_norm:
                best_solution = xi_trial
                best_norm = norm_trial
                best_success = solution.success
                best_physical = True

        if best_solution is None:

            solution = root(
                reaction_extent_residual,
                xi_guess,
                args=(n_in_equilibrium_optimal, T_val, pressure),
                method="lm"
            )

            best_solution = solution.x

            best_norm = np.linalg.norm(
                reaction_extent_residual(
                    best_solution,
                    n_in_equilibrium_optimal,
                    T_val,
                    pressure
                )
            )

            best_success = solution.success
            best_physical = False

        xi_optimal[:, p_idx, T_idx] = best_solution
        residual_norm_optimal[p_idx, T_idx] = best_norm
        solver_success_optimal[p_idx, T_idx] = best_success
        physical_solution_optimal[p_idx, T_idx] = best_physical

        # Continuation: use previous solution as next initial guess
        xi_guess = best_solution


# Validation
valid_solution_optimal = (
    solver_success_optimal
    & physical_solution_optimal
    & (residual_norm_optimal < 1e-6)
)

xi_optimal_plot = xi_optimal.copy()

for p_idx in range(len(p_equilibrium)):
    for T_idx in range(len(T_equilibrium)):
        if not valid_solution_optimal[p_idx, T_idx]:
            xi_optimal_plot[:, p_idx, T_idx] = np.nan


# Calculate outlet molar flows and mole fractions
for p_idx, pressure in enumerate(p_equilibrium):

    for T_idx, T_val in enumerate(T_equilibrium):

        if valid_solution_optimal[p_idx, T_idx]:

            xi_current = xi_optimal[:, p_idx, T_idx]

            n_out_current, x_out_current = composition_from_extent(
                xi_current,
                n_in_equilibrium_optimal
            )

            n_out_optimal[:, p_idx, T_idx] = n_out_current
            x_out_optimal[:, p_idx, T_idx] = x_out_current


print("Optimal inlet reaction extent calculation finished.")
print(f"Successful solver calls: {solver_success_optimal.sum()} / {solver_success_optimal.size}")
print(f"Valid physical equilibrium points: {valid_solution_optimal.sum()} / {valid_solution_optimal.size}")

valid_residuals_optimal = np.where(
    valid_solution_optimal,
    residual_norm_optimal,
    np.nan
)

print(
    f"Maximum residual norm of valid points: "
    f"{np.nanmax(valid_residuals_optimal):.3e}"
)

for j, reaction_name in enumerate(independent_reactions):
    print(
        f"{reaction_name}: "
        f"xi range = {np.nanmin(xi_optimal_plot[j]):.3e} "
        f"to {np.nanmax(xi_optimal_plot[j]):.3e} mol/s"
    )


# Plot reaction extents for optimal inlet composition

equilibrium_pressure_colors = {
    pressure: plt.cm.tab10(i % 10)
    for i, pressure in enumerate(p_equilibrium)
}

fig, axes = plt.subplots(
    1,
    R_nu,
    figsize=(5.5 * R_nu, 4.5),
    sharex=True
)

axes = np.atleast_1d(axes)

for j, reaction_name in enumerate(independent_reactions):

    ax = axes[j]

    for p_idx, pressure in enumerate(p_equilibrium):

        ax.plot(
            T_equilibrium - 273.15,
            xi_optimal_plot[j, p_idx, :],
            label=f"{pressure:.0f} bar",
            color=equilibrium_pressure_colors[pressure]
        )

    ax.set_title(f"Optimal inlet - {reaction_name}")
    ax.set_xlabel("Temperature / °C")
    ax.set_ylabel(r"Reaction extent $\xi$ / mol s$^{-1}$")
    ax.grid(True)
    ax.legend(title="Pressure", fontsize=8)

fig.suptitle(
    rf"Reaction extents for optimal inlet composition "
    rf"$(H_2/CO_2 = {optimal_H2_CO2_ratio:.2f})$",
    fontsize=16,
    y=1.05
)

plt.tight_layout()
plt.show()